# MLP em PyTorch com Loop de Treinamento, Early Stopping e Batching

## Objetivo
Implementar um **loop de treinamento explícito** com **validação**, **early stopping** (paciência sobre loss de validação) e **batching** organizado via `DataLoader`. Este notebook estende o trabalho do notebook `03_mlp_pytorch.ipynb` com **práticas realistas e reprodutíveis** de engenharia de ML.

### Por que esta etapa?
No notebook anterior (`03_mlp_pytorch.ipynb`) treinamos um MLP de forma simples. Aqui, refinamos essa abordagem com:
- **Split treino/validação/teste:** Mais rigoroso na avaliação de generalização.
- **Loop explícito:** Controle fino sobre cada epoch e possibilidade de logging.
- **Early stopping:** Para o treinamento quando a validação não melhora, economizando tempo e evitando overfitting.
- **Batching organizado:** Uso consciente de `DataLoader` com shuffle no treino.
- **MLflow:** Rastreabilidade de parâmetros, métricas e modelo final.

## O que será feito (passo a passo) e por quê

1. Carregar dados do EDA (`Telco_customer_churn_ready.csv`) — dados já limpos e codificados.
2. Dividir em **treino (60%), validação (15%) e teste (25%)** com estratificação — manter proporção de churn em cada split.
3. Padronizar features com `StandardScaler` **apenas ajustado no treino** — evitar vazamento de informação.
4. Criar `DataLoader` com batching — eficiência computacional e regularização.
5. Definir MLP com `torch.nn.Module` — arquitetura repeatable com ReLU e BCEWithLogitsLoss.
6. Implementar loop de treinamento e validação — iteração época por época com feedback.
7. Adicionar **early stopping** com paciência sobre loss de validação — parar no tempo certo.
8. Avaliar no **conjunto de teste** com ≥4 métricas — acurácia, precisão, recall, F1, ROC-AUC.
9. Registrar tudo no **MLflow** — rastreabilidade de parâmetros, métricas e modelo.
10. Gerar visualizações (loss curves, matriz de confusão) — interpretabilidade do treinamento.

In [1]:
from pathlib import Path
from copy import deepcopy
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de seeds para reprodutibilidade.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) if torch.cuda.is_available() else None

# Verificar se GPU está disponível (informativo).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

# Procurar o arquivo de dados tratado (mesmo padrão do notebook 02).
READY_PATH_CANDIDATES = [
    Path("../data/Telco_customer_churn_ready.csv"),
    Path("data/Telco_customer_churn_ready.csv"),
]
DATA_PATH = next((p for p in READY_PATH_CANDIDATES if p.exists()), None)

assert DATA_PATH is not None, (
    "Arquivo tratado do EDA não encontrado. "
    "Rode o notebook de EDA para gerar Telco_customer_churn_ready.csv."
)
print(f"Arquivo de entrada encontrado: {DATA_PATH.resolve()}")

Dispositivo: cpu
Arquivo de entrada encontrado: C:\Users\eduardo.silva\Downloads\Projeto_Churn\data\Telco_customer_churn_ready.csv


## Carregamento e Exploração dos Dados

In [ ]:
# Carrega o dataset já tratado no EDA.
TARGET = "Churn Value"
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nPrimeiras colunas: {df.columns.tolist()[:10]}")

# Verifica a coluna alvo.
assert TARGET in df.columns, f"Coluna alvo '{TARGET}' não encontrada."
print(f"\nDistribuição do alvo (proporção):\n{df[TARGET].value_counts(normalize=True)}")

display(df.head(3))

In [ ]:
# Separa features (X) e alvo (y).
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Converte para numpy com float32 (compatível com PyTorch e StandardScaler).
X_np = X.astype(np.float32).to_numpy()
y_np = y.to_numpy(dtype=np.float32)

# Split 1: Treino+Validação (80%) vs Teste (20%) — com estratificação.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_np,
    y_np,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

# Split 2: Treino (75% de 80% = 60%) vs Validação (25% de 80% = 20%).
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp,
)

n_features = X_train.shape[1]
print(f"Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")
print(f"Total de features: {n_features}")

In [ ]:
# Padroniza features com StandardScaler — FIT APENAS NO TREINO (evita vazamento).
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

print("Padronização concluída (sem vazamento de validação/teste para treino).")
print(f"Média (treino) — primeiras 3 features: {X_train_s.mean(axis=0)[:3]}")
print(f"Desvio (treino) — primeiras 3 features: {X_train_s.std(axis=0)[:3]}")

## Criação de DataLoaders com Batching

In [ ]:
# Hiperparâmetros de treinamento.
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS_MAX = 50  # Máximo; early stopping pode parar antes.
PATIENCE = 10  # Quantos epochs sem melhoria antes de parar.
THRESHOLD_PREDICTION = 0.5  # Threshold para converter prob → classe.

# Cria TensorDataset para cada split.
train_ds = TensorDataset(
    torch.from_numpy(X_train_s),
    torch.from_numpy(y_train).unsqueeze(1),  # Shape: (n, 1)
)
val_ds = TensorDataset(
    torch.from_numpy(X_val_s),
    torch.from_numpy(y_val).unsqueeze(1),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test_s),
    torch.from_numpy(y_test).unsqueeze(1),
)

# Cria DataLoaders.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## Definição da Arquitetura MLP

In [ ]:
class MLPChurn(nn.Module):
    """
    Perceptron multicamadas para classificação binária de churn.
    
    Arquitetura:
    - Entrada: input_dim features
    - Camada oculta 1: 64 neurônios + ReLU
    - Camada oculta 2: 32 neurônios + ReLU
    - Saída: 1 neurônio (logit, sem ativação)
    """

    def __init__(self, input_dim: int, hidden_dim_1: int = 64, hidden_dim_2: int = 32):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x


# Instancia o modelo.
model = MLPChurn(input_dim=n_features).to(DEVICE)
print(f"Modelo instanciado no dispositivo: {DEVICE}")
print(model)

## Definição de Loss e Otimizador

In [ ]:
# BCEWithLogitsLoss: combina sigmoid + binary cross-entropy de forma numericamente estável.
# A saída da rede são logits; a loss aplica sigmoid internamente.
criterion = nn.BCEWithLogitsLoss()

# Otimizador Adam: taxa adaptativa e funciona bem em muitos casos.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss function: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__} (lr={LEARNING_RATE})")

## Funções de Treinamento e Avaliação

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Treina o modelo por um epoch completo.
    
    Args:
        model: instância do modelo.
        dataloader: DataLoader com dados de treino.
        criterion: função de loss.
        optimizer: otimizador.
        device: 'cuda' ou 'cpu'.
    
    Returns:
        avg_loss: loss médio do epoch.
    """
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass.
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        # Backward pass.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss


def evaluate(model, dataloader, criterion, device):
    """
    Avalia o modelo em um conjunto de dados.
    
    Args:
        model: instância do modelo.
        dataloader: DataLoader com dados para avaliar.
        criterion: função de loss.
        device: 'cuda' ou 'cpu'.
    
    Returns:
        avg_loss: loss médio.
        y_true: labels verdadeiros (numpy).
        y_proba: probabilidades preditas (numpy).
    """
    model.eval()
    total_loss = 0.0
    y_true_list = []
    y_proba_list = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)

            # Converte logits para probabilidades.
            proba = torch.sigmoid(logits).cpu().numpy()
            y_true_list.append(y_batch.cpu().numpy())
            y_proba_list.append(proba)

    avg_loss = total_loss / len(dataloader.dataset)
    y_true = np.vstack(y_true_list).flatten()
    y_proba = np.vstack(y_proba_list).flatten()

    return avg_loss, y_true, y_proba


print("Funções de treinamento e avaliação definidas.")

In [ ]:
# Histórico de treinamento.
history = {"train_loss": [], "val_loss": []}

# Rastreamento para early stopping.
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

print(f"Iniciando treinamento (max_epochs={NUM_EPOCHS_MAX}, patience={PATIENCE})...\n")

for epoch in range(NUM_EPOCHS_MAX):
    # Treina por um epoch.
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)

    # Avalia em validação.
    val_loss, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    # Registra histórico.
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Verifica se validação melhorou.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = deepcopy(model.state_dict())
        status = "[MELHOR] ✓"
    else:
        patience_counter += 1
        status = f"Paciência: {patience_counter}/{PATIENCE}"

    # Print informativo (a cada epoch para feedback contínuo).
    print(
        f"Epoch {epoch+1:3d}/{NUM_EPOCHS_MAX} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | {status}"
    )

    # Early stopping.
    if patience_counter >= PATIENCE:
        print(f"\n✓ Early stopping acionado no epoch {epoch+1}.")
        break

# Carrega o melhor modelo encontrado.
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nMelhor modelo carregado (val_loss={best_val_loss:.6f}).")

## Avaliação no Conjunto de Teste

In [ ]:
# Avalia no teste com o melhor modelo.
test_loss, y_test_true, y_test_proba = evaluate(model, test_loader, criterion, DEVICE)

# Converte probabilidades em predições (threshold = 0.5).
y_test_pred = (y_test_proba >= THRESHOLD_PREDICTION).astype(int)

# Calcula métricas (conforme docs/METRICAS.md).
accuracy = accuracy_score(y_test_true, y_test_pred)
precision = precision_score(y_test_true, y_test_pred, zero_division=0)
recall = recall_score(y_test_true, y_test_pred, zero_division=0)
f1 = f1_score(y_test_true, y_test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test_true, y_test_proba)

print("="*70)
print("AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*70)
print(f"Test Loss: {test_loss:.6f}")
print(f"\nMétricas (threshold={THRESHOLD_PREDICTION}):")
print(f"  Acurácia:  {accuracy:.4f}")
print(f"  Precisão:  {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print("="*70)

## Visualização: Evolução da Loss

In [ ]:
# Plot: Loss ao longo do treinamento.
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    history["train_loss"],
    label="Train Loss",
    marker="o",
    linestyle="-",
    markersize=4,
    alpha=0.7,
)
ax.plot(
    history["val_loss"],
    label="Val Loss",
    marker="s",
    linestyle="-",
    markersize=4,
    alpha=0.7,
)
ax.axvline(
    x=len(history["train_loss"]) - 1,
    color="red",
    linestyle="--",
    alpha=0.5,
    label="Early Stopping",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Evolução da Loss Durante o Treinamento (Early Stopping)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

n_epochs_trained = len(history["train_loss"])
print(f"Treinamento completado em {n_epochs_trained} epochs.")

## Visualização: Matriz de Confusão

In [ ]:
# Calcula matriz de confusão.
cm = confusion_matrix(y_test_true, y_test_pred)

# Plot: Matriz de confusão.
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    cbar=True,
    xticklabels=["Não Churn", "Churn"],
    yticklabels=["Não Churn", "Churn"],
)
ax.set_xlabel("Predito")
ax.set_ylabel("Verdadeiro")
ax.set_title("Matriz de Confusão (Conjunto de Teste)")
plt.tight_layout()
plt.show()

# Extrai componentes da matriz.
tn, fp, fn, tp = cm.ravel()
print(f"\nComponentes da Matriz de Confusão:")
print(f"  TN (Verdadeiro Negativo):  {tn}")
print(f"  FP (Falso Positivo):       {fp}")
print(f"  FN (Falso Negativo):       {fn}")
print(f"  TP (Verdadeiro Positivo):  {tp}")

## Integração com MLflow

In [ ]:
# Força o tracking para banco local e evita problemas de caminho.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

# Hash do dataset para rastreabilidade.
def file_sha256(path: Path) -> str:
    hash_obj = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

dataset_version = file_sha256(DATA_PATH)
dataset_name = DATA_PATH.name

# Inicia experimento no MLflow.
mlflow.set_experiment("MLP-Churn-EarlyStoppingBatching")

with mlflow.start_run(run_name="04_mlp_training_early_stopping"):
    # Registra parâmetros de configuração.
    mlflow.log_param("seed", SEED)
    mlflow.log_param("dataset_name", dataset_name)
    mlflow.log_param("dataset_version_sha256", dataset_version)
    mlflow.log_param("n_amostras_total", int(df.shape[0]))
    mlflow.log_param("n_features", int(n_features))
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("num_epochs_max", NUM_EPOCHS_MAX)
    mlflow.log_param("patience_early_stopping", PATIENCE)
    mlflow.log_param("threshold_prediction", THRESHOLD_PREDICTION)
    mlflow.log_param("hidden_dim_1", 64)
    mlflow.log_param("hidden_dim_2", 32)
    mlflow.log_param("loss_function", "BCEWithLogitsLoss")
    mlflow.log_param("optimizer", "Adam")

    # Registra métricas finais.
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_precision", precision)
    mlflow.log_metric("test_recall", recall)
    mlflow.log_metric("test_f1_score", f1)
    mlflow.log_metric("test_roc_auc", roc_auc)
    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_metric("epochs_trained", n_epochs_trained)
    mlflow.log_metric("confusion_tn", tn)
    mlflow.log_metric("confusion_fp", fp)
    mlflow.log_metric("confusion_fn", fn)
    mlflow.log_metric("confusion_tp", tp)

    # Registra o modelo.
    mlflow.pytorch.log_model(model, "model")

print("\\n✓ Experimento registrado no MLflow!")
print(f"ID do Run: {mlflow.active_run().info.run_id}")
print(f"Experimento: {mlflow.active_run().info.experiment_id}")

## Resumo Final do Experimento

In [ ]:
print("\\n" + "="*80)
print("RESUMO — MLP com Loop de Treinamento, Early Stopping e Batching")
print("="*80)
print(f"\\nDataset: {DATA_PATH.name}")
print(f"Seed para reprodutibilidade: {SEED}")
print(f"\\nSplits:")
print(f"  Treino:      {len(X_train)} amostras (60%)")
print(f"  Validação:   {len(X_val)} amostras (15%)")
print(f"  Teste:       {len(X_test)} amostras (25%)")
print(f"  Features:    {n_features}")
print(f"\\nArquitetura MLP:")
print(f"  Input ({n_features}) → 64 (ReLU) → 32 (ReLU) → 1 (Logit)")
print(f"\\nHiperparâmetros:")
print(f"  Batch Size:                {BATCH_SIZE}")
print(f"  Learning Rate:             {LEARNING_RATE}")
print(f"  Loss Function:             BCEWithLogitsLoss")
print(f"  Optimizer:                 Adam")
print(f"  Early Stopping Patience:   {PATIENCE}")
print(f"  Epochs Treinados:          {n_epochs_trained}/{NUM_EPOCHS_MAX}")
print(f"\\nResultados no Teste (threshold={THRESHOLD_PREDICTION}):")
print(f"  Loss:       {test_loss:.6f}")
print(f"  Acurácia:   {accuracy:.4f}")
print(f"  Precisão:   {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print(f"  ROC-AUC:    {roc_auc:.4f}")
print(f"\\nMatriz de Confusão (Teste):")
print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"\\nPróximas etapas:")
print(f"  → Comparar com baselines (05_compare_mlp_baselines.ipynb)")
print(f"  → Analisar trade-offs FP vs FN (06_tradeoff_custo_fp_fn.ipynb)")
print(f"  → Consolidar experimentos (07_mlflow_experimentos_mlp_ensembles.ipynb)")
print("="*80)